In [1]:
import os

import numpy as np
import pandas as pd

import torch
from torch import nn

In [2]:
import sys, os
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.tfa import TFA
from scLEMBAS.model.train import TrainSimple, TrainCat

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888

device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')
if not os.path.isdir(models_path):
    os.mkdir(models_path)

In [4]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'processed', 'ID_tf_activity.h5ad'))

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.4.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [6]:
from scLEMBAS.io import write_pickled_object, read_pickled_object
# sn_ppis = parse_network.load_network('omnipath', organism = 'mouse', static = True)
# sn_ppis = parse_network.extract_network(sn_ppis, curation_effort_thresh = 5, n_references_thresh = 3,
#                                         resources = ['HuRI','IntAct','KEGG-MEDICUS','NetPath','Reactome_SignaLink3','SPIKE','SignaLink3','SIGNOR', 
#                                                 'Baccin2019', 'Ramilowski2015', 'Reactome_LRdb', 'UniProt_LRdb', 'CellChatDB', 'CellPhoneDB', 'connectomeDB2020', 'scConnect'], 
#                                         drop_self = True, verbose = True)

# tf_labels = tf_adata.var.index.unique().tolist()

# ligand_labels = tf_adata.obs['sample'].unique().tolist()
# ligand_labels = [(l[0] + l[1:].lower()).replace('-', '') for l in ligand_labels] # mouse naming convention

# # filter for paths b/w ligand and tf
# fn_1 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
#                        path_finder = 'shortest')
# fn_2 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
#                        path_finder = 'connected')
# # of the methods to identify paths, retain the one that has the most interactions
# if fn_1.shape[0] > fn_2.shape[0]:
#     sn_ppis = fn_1
# else:
#     sn_ppis = fn_2

# del fn_1, fn_2

# sn_ppis.loc[sn_ppis[(sn_ppis[stimulation_label] == 1) & (sn_ppis[inhibition_label] == 1)].index, 
#     [stimulation_label, inhibition_label]] = [False, False]
# sn_ppis = parse_network.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label)

# all_nodes = sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()
# input_ligands_available = sorted(set(ligand_labels).intersection(all_nodes))

# subset_tf = tf_adata[tf_adata.obs.TF_clusters.isin(['9', '15'])]
# sample_size = subset_tf.obs.TF_clusters.value_counts().min()
# large_cluster = subset_tf.obs.TF_clusters.value_counts().idxmax()
# small_cluster = subset_tf.obs.TF_clusters.value_counts().idxmin()
# large_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == large_cluster].index
# small_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == small_cluster].index.tolist()
# np.random.seed(seed)
# lcb_sub = list(np.random.choice(large_cluster_barcodes, sample_size, replace = False))
# subset_tf = subset_tf[lcb_sub + small_cluster_barcodes, :]
# subset_tf.obs.TF_clusters.value_counts()

# np.random.seed(seed)
# selected_ligand = np.random.choice(input_ligands_available, 1)[0]

# ligand_input = pd.DataFrame(subset_tf.obs.TF_clusters.map({'9': 0, '15': 1}))
# ligand_input.columns = [selected_ligand]
# tf_output = pd.DataFrame(subset_tf.X, index = subset_tf.obs.index, columns = subset_tf.var.index)

# mod_in = [sn_ppis, ligand_input, tf_output, subset_tf]
# write_pickled_object(mod_in, 'trash.pickle')
mod_in = read_pickled_object('trash.pickle')
sn_ppis, ligand_input, tf_output, subset_tf = mod_in

In [7]:
# linear scaling of inputs/outputs
projection_amplitude_in = 3
projection_amplitude_out = 1.2
# other parameters
bionet_params = {'target_steps': 100, 
                 'max_steps': 120, 
                 'exp_factor':50, 
                 'tolerance': 1e-5, 
                 'leak':1e-2, 
                'cat_max_norm': 1} # fed directly to model

# training parameters
lr_params = {'max_epochs': 100, 
             'learning_rate': 2e-3, 
             'reset_optimizer_epoch': 200}

other_params = {'batch_size': 256, 
                'network_noise_scale': 10, 
                'gradient_noise_scale': 1e-9}

regularization_params = {'param_lambda_L2': 1e-6, 
                         'discriminator_lambda_L2': 1e-5,
                         'moa_lambda_L1': 0.1, #'ligand_lambda_L2': 1e-5, 
                         'uniform_lambda_L2': 1e-4,  
                         'uniform_max': (1/1.2), 
                         'spectral_loss_factor': 1e-5}

spectral_radius_params = {'n_probes_spectral': 5, 
                          'power_steps_spectral': 50, 
                          'subset_n_spectral': 10}
training_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}
target_spectral_radius = 0.8

# parameters for constructing the discriminator
discriminator_params = {'batch_momentum': 0.01,
                        'layer_norm': False,
                        'dropout_rate': 0.1,
                        'activation_fn': torch.nn.modules.activation.ReLU,
                        'n_hidden_nodes': [16, 16, 16],
                        'optimizer': torch.optim.Adam}

# # TFA
# encoder_dist = None
# decoder_dist = 'gauss'

# e_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
# d_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
# ve_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}
# vd_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}

# Hyper_params = {
#     'tfa': {'n_latent': 32, 'cat_max_norm': 1, 'encoder_dist': encoder_dist, 'decoder_dist': decoder_dist}, 
#     'encoder': ve_hp if encoder_dist == 'guass' else e_hp,
#     'decoder': vd_hp if decoder_dist == 'guass' else d_hp,
#     'discriminator': {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 'activation_fn': nn.ReLU, 'n_hidden_nodes': [16, 16, 16], 'return_logits': True}

# }

In [8]:
cat_keys = ['TF_clusters', 'celltype']
mod = SignalingModel(net = sn_ppis,
                     X_in = ligand_input,
                     y_out = tf_output, 
                     activation_function = 'MML', 
                     covariates = subset_tf.obs,
                     categorical_covariate_keys = cat_keys,
                     projection_amplitude_in = projection_amplitude_in, projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params,
                     dtype = torch.float32, device = device, seed = seed)

# Start dev

In [9]:
# # define inputs
# X_in = mod.df_to_tensor(mod.X_in)
# covariates_idx = mod.signaling_network.covariates_to_tensor(sample_ids = mod.X_in.index)

# X_full = mod.input_layer(X_in) # transform to full network with ligand input concentrations
# Y_full = mod.signaling_network(X_full, covariates_idx) # train signaling network weights
# Y_hat = mod.output_layer(Y_full)

In [10]:
trainer = TrainCat(mod = mod,
                   prediction_optimizer = torch.optim.Adam,
                   prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
                   hyper_params = training_params,
                   discriminator_params = discriminator_params,
                   train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
                   train_seed = seed)
self = trainer

/home/hmbaghda/Projects/scLEMBAS/scLEMBAS/model/train.py:115: UserWarning: Recommended to run `self.mod.signaling_network.prescale_weights()` prior to training
  warnings.warn('Recommended to run `self.mod.signaling_network.prescale_weights()` prior to training')


In [11]:
from scLEMBAS.model.train import *

In [12]:
e = 0
# set learning rate
cur_lr = utils.get_lr(e, self.hyper_params['max_epochs'], max_height = self.hyper_params['learning_rate'],
                    start_height=self.hyper_params['learning_rate']/10, end_height=1e-6, peak = 1000)
self.prediction_optimizer.param_groups[0]['lr'] = cur_lr

cur_loss = []
cur_eig = []

# iterate through batches
if self.mod.seed:
    utils.set_seeds(self.mod.seed + e)
    
for batch, (X_in_, y_out_, covariates_idx_) in enumerate(self.train_dataloader):
    break

In [ ]:
cur_lr = utils.get_lr(e, self.hyper_params['max_epochs'], max_height = self.hyper_params['learning_rate'],
                    start_height=self.hyper_params['learning_rate']/10, end_height=1e-6, peak = 1000)

self.prediction_optimizer.param_groups[0]['lr'] = cur_lr
for dopt in self.discriminator_optimizers.values():
    dopt.param_groups[0]['lr'] = cur_lr

cur_loss = []
cur_eig = []

# iterate through batches
if self.mod.seed:
    utils.set_seeds(self.mod.seed + e)
for batch, (X_in_, y_out_, covariates_idx_) in enumerate(self.train_dataloader):
    self.mod.train()
    self.prediction_optimizer.zero_grad()

    X_in_, y_out_, covariates_idx_ = X_in_.to(self.mod.device), y_out_.to(self.mod.device), covariates_idx_.to(self.mod.device)
    covariates_idx_ = covariates_idx_.to(self.mod.device)

In [18]:
self.mod.train()
self.prediction_optimizer.zero_grad()
for dopt in self.discriminator_optimizers.values():
    dopt.zero_grad()


X_in_, y_out_, covariates_idx_ = X_in_.to(self.mod.device), y_out_.to(self.mod.device), covariates_idx_.to(self.mod.device)
covariates_idx_ = covariates_idx_.to(self.mod.device)

# forward pass - gives the TF activity prediction and generates the basal_bias
X_full = self.mod.input_layer(X_in_) # transform to full network with ligand input concentrations
utils.set_seeds(self.mod.seed + self.mod._gradient_seed_counter)
network_noise = torch.randn(X_full.shape, device = X_full.device)
X_full = X_full + (self.hyper_params['network_noise_scale'] * cur_lr * network_noise) # randomly add noise to signaling network input, makes model more robust
Y_full = self.mod.signaling_network(X_full, covariates_idx_) # train signaling network weights
Y_hat = self.mod.output_layer(Y_full)

prediction_loss = self.prediction_loss_fn(y_out_, Y_hat)

In [20]:
Y_full

tensor([[ 0.0552,  0.0653, -0.0014,  ..., -0.0004, -0.0004,  0.0235],
        [ 0.0026, -0.0010, -0.0003,  ..., -0.0003,  0.0780,  0.0718],
        [ 0.0516,  0.0622, -0.0014,  ..., -0.0004, -0.0005,  0.0184],
        ...,
        [ 0.1269,  0.0695,  0.0716,  ..., -0.0012,  0.0832,  0.0723],
        [-0.0004, -0.0010,  0.1213,  ...,  0.0611, -0.0009,  0.0320],
        [-0.0004, -0.0009,  0.1152,  ...,  0.0587, -0.0009,  0.0317]],
       device='cuda:0', grad_fn=<PermuteBackward0>)

In [14]:
self.mod.train()
self.prediction_optimizer.zero_grad()
for dopt in self.discriminator_optimizers.values():
    dopt.zero_grad()


X_in_, y_out_, covariates_idx_ = X_in_.to(self.mod.device), y_out_.to(self.mod.device), covariates_idx_.to(self.mod.device)
covariates_idx_ = covariates_idx_.to(self.mod.device)

# forward pass - gives the TF activity prediction and generates the basal_bias
X_full = self.mod.input_layer(X_in_) # transform to full network with ligand input concentrations
utils.set_seeds(self.mod.seed + self.mod._gradient_seed_counter)
network_noise = torch.randn(X_full.shape, device = X_full.device)
X_full = X_full + (self.hyper_params['network_noise_scale'] * cur_lr * network_noise) # randomly add noise to signaling network input, makes model more robust
Y_full = self.mod.signaling_network(X_full, covariates_idx_) # train signaling network weights
Y_hat = self.mod.output_layer(Y_full)

prediction_loss = self.prediction_loss_fn(y_out_, Y_hat)


# discriminator prediction
discriminator_loss = torch.tensor([0], device = self.mod.device, dtype = self.mod.dtype)
for cat_group_idx, (cat, discriminator) in enumerate(self.discriminators.items()):
    bias_basal_prediction = discriminator(self.mod.signaling_network.bias_basal.T) # predicted logits
    # TO DO: should this expansion of the vector into samples be done after getting the prediction or 
    # should it be done prior to 
    # if prior, should the batch_norm = False line in the initilize_discriminator be removed?
    bias_basal_prediction = bias_basal_prediction.repeat(covariates_idx_.shape[0], 1) # expand vector to # of samples

    target = covariates_idx_[:, cat_group_idx]
    if discriminator.n_labels == 2:
        target = target.to(self.mod.dtype).unsqueeze(1)

    discriminator_loss += discriminator.loss_fn(bias_basal_prediction, target)
    # TO DO: does it need to be recalculated like this?
    prediction_loss -= discriminator.loss_fn(bias_basal_prediction, target) 


discriminator_loss.backward(retain_graph = True)
prediction_loss.backward()
self.mod.add_gradient_noise(noise_level = self.hyper_params['gradient_noise_scale'])
self.prediction_optimizer.step()
for dopt in self.discriminator_optimizers.values():
    dopt.step()

In [ ]:
   def L2Regularization(self, L2):

        

In [50]:
discriminator.classifier[0].fc_layers

Sequential(
  (FC Layer 0): Sequential(
    (linear): Linear(in_features=332, out_features=16, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (FC Layer 1): Sequential(
    (linear): Linear(in_features=16, out_features=16, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (FC Layer 2): Sequential(
    (linear): Linear(in_features=16, out_features=16, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [51]:
regularization_loss = 0


tensor(21.3243, device='cuda:0', grad_fn=<AddBackward0>)

In [37]:
discriminator.classifier.parameters()

<generator object Module.parameters at 0x200154d33840>

In [28]:
cat_group_idx = 0
cat = 'TF_clusters'

# cat_group_idx = 1
# cat = 'celltype'

discriminator = self.discriminators[cat]

In [29]:
bias_basal_prediction = discriminator(self.mod.signaling_network.bias_basal.T) # predicted logits
# TO DO: should this expansion of the vector into samples be done after getting the prediction or 
# should it be done prior to 
# if prior, should the batch_norm = False line in the initilize_discriminator be removed?
bias_basal_prediction = bias_basal_prediction.repeat(covariates_idx_.shape[0], 1) # expand vector to # of samples
target = covariates_idx_[:, cat_group_idx]
if discriminator.n_labels == 2:
    target = target.to(self.mod.dtype).unsqueeze(1)
self.discriminator_losses[cat](bias_basal_prediction,
                               target
                          )



tensor(0.6871, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

In [18]:
covariates_idx_[:, cat_group_idx].to(self.mod.dtype).shape

torch.Size([256])

In [19]:
bias_basal_prediction.shape

torch.Size([256, 1])

In [55]:
loss = nn.BCEWithLogitsLoss()
predicted = torch.randn(3, requires_grad=True)
target = torch.empty(3).random_(2)
output = loss(predicted.unsqueeze(1), target.unsqueeze(1))

In [66]:
self.mod.signaling_network.covariates_idx

,TF_clusters,celltype
AGGCCACCAAGTTTGC-35,1,8
ATCTTCAAGAGCTTTC-37,1,9
GGTGAAGCAGCAGGAT-02,1,7
CAGCCAGCAGCCATTA-11,1,6
AGACACTAGTCGAAAT-28,1,2
...,...,...
CTGTCGTAGTCCCGAC-04,0,5
ACGGAAGCAGACCTAT-04,0,5
TGGGCTGCATAGGAGC-40,0,5
TTACCATCACCAGCGT-04,0,5


In [63]:
covariates_idx_[:, cat_group_idx].to(mod.dtype)

tensor([0., 1., 0., 1., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0.,
        0., 1., 0., 0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 1., 0., 1.,
        0., 1., 1., 1., 1., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0., 1., 1.,
        0., 1., 0., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1.,
        0., 0., 1., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 1., 0., 0., 1., 0.,
        0., 0., 1., 0., 1., 0., 0., 1., 0., 1., 0., 1., 0., 1., 1., 1., 0., 0.,
        0., 0., 1., 0., 0., 1., 0., 0., 1., 1., 1., 0., 0., 1., 0., 0., 1., 1.,
        1., 1., 0., 0., 0., 1., 0., 0., 1., 1., 1., 0., 0., 1., 1., 1., 1., 0.,
        1., 1., 0., 1., 0., 0., 0., 1., 1., 0., 0., 1., 0., 1., 1., 1., 1., 0.,
        0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 1., 0., 0., 0., 1.,
        0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0.,
        1., 1., 0., 0., 0., 1., 0., 1., 

In [55]:
input = torch.randn(3, requires_grad=True)
target = torch.empty(3).random_(2)

In [58]:
input = torch.randn(3, 5, requires_grad=True)
target = torch.empty(3, dtype=torch.long).random_(5)

In [62]:
input.shape

torch.Size([3, 5])

In [63]:
target

tensor([1, 0, 3])

In [18]:
n_features_in = 332
n_labels = 2
n_hidden_nodes = [16, 16, 16]
batch_momentum = 0.01
layer_norm = False
dropout_rate = 0.1
activation_fn = nn.ReLU
optimizer = torch.optim.Adam
dtype = torch.float32
device = mod.device

from scLEMBAS.model.model_components import *

class Test(nn.Module):
    def __init__(self):
        super().__init__()
        pass
    def forward(self):
        pass
    
self = Test()

In [19]:
self.n_labels = n_labels
if self.n_labels > 2: # multi-class
    self.loss_fn = nn.CrossEntropyLoss # applies softmax to logits prior to CE
    out_features = self.n_labels
elif self.n_labels == 2: # binary
    self.loss_fn = nn.BCEWithLogitsLoss # applies sigmoid to logits prior to CE
    out_features = 1
else:
    raise ValueError('There are no distinct classes.')
    
self.optimizer = optimizer

In [64]:
self.test = FCLayers(layers = [n_features_in] + n_hidden_nodes, 
                           batch_momentum = batch_momentum, 
                           layer_norm = layer_norm, 
                           dropout_rate = dropout_rate, 
                           activation_fn = activation_fn, 
                           dtype = dtype, device = device)

In [56]:
a = self.test.fc_layers[0].linear(torch.randn(332,1).to(mod.device).T)

In [67]:
mod.signaling_network.bias_basal.shape

torch.Size([332, 1])

In [63]:
mod.signaling_network.bias_basal

Parameter containing:
tensor([[0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
        [0.0010],
      

In [61]:
FCLayers.DEFAULT_HYPER_PARAMS

{'layer_norm': False,
 'dropout_rate': 0.1,
 'activation_fn': torch.nn.modules.activation.ReLU}

In [58]:
self.test.fc_layers[0][1](a)

tensor([[0.0000, 0.9132, 0.0000, 0.0000, 0.0564, 0.4983, 0.0000, 0.0000, 0.0000,
         0.3727, 0.6668, 0.4955, 0.0000, 0.0863, 0.2272, 0.0000]],
       device='cuda:0', grad_fn=<ReluBackward0>)

In [29]:
self.test.fc_layers[0].linear

Linear(in_features=332, out_features=16, bias=True)

In [21]:
self.test(torch.randn(332,1).to(mod.device))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (332x1 and 332x16)

In [ ]:

cat_layers = []
cat_layers.append(FCLayers(layers = [n_features_in] + n_hidden_nodes, 
                           batch_momentum = batch_momentum, 
                           layer_norm = layer_norm, 
                           dropout_rate = dropout_rate, 
                           activation_fn = activation_fn, 
                           dtype = dtype, device = device))
cat_layers.append(FCLayers(layers = [n_hidden_nodes[-1], out_features], 
                           dtype = dtype, device = device,
                           batch_momentum = None, layer_norm = False, dropout_rate = None, activation_fn = None))

self.classifier = nn.Sequential(*cat_layers)

In [31]:
discriminator(torch.randn(332, 1).to(mod.device).T)

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 16])

In [20]:
discriminator = self.discriminators['TF_clusters']
discriminator(self.mod.signaling_network.bias_basal)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (332x1 and 332x16)

In [36]:
adverserial_loss = 

In [39]:
discriminator

CatDiscriminator(
  (classifier): Sequential(
    (0): FCLayers(
      (fc_layers): Sequential(
        (FC Layer 0): Sequential(
          (linear): Linear(in_features=332, out_features=16, bias=True)
          (batch normalization): BatchNorm1d(16, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (FC Layer 1): Sequential(
          (linear): Linear(in_features=16, out_features=16, bias=True)
          (batch normalization): BatchNorm1d(16, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (FC Layer 2): Sequential(
          (linear): Linear(in_features=16, out_features=16, bias=True)
          (batch normalization): BatchNorm1d(16, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=

In [20]:


        # get prediction loss
        prediction_loss = self.prediction_loss_fn(y_out_, Y_hat)

        # get regularization losses
        sign_reg = self.mod.signaling_network.sign_regularization(lambda_L1 = self.hyper_params['moa_lambda_L1']) # incorrect MoA
#             ligand_reg = self.mod.ligand_regularization(lambda_L2 = self.hyper_params['ligand_lambda_L2']) # ligand biases
        stability_loss, spectral_radius = self.mod.signaling_network.get_SS_loss(Y_full = Y_full.detach(), spectral_loss_factor = self.hyper_params['spectral_loss_factor'],
                                                                            subset_n = self.hyper_params['subset_n_spectral'], n_probes = self.hyper_params['n_probes_spectral'], 
                                                                            power_steps = self.hyper_params['power_steps_spectral'])
        uniform_reg = self.mod.uniform_regularization(lambda_L2 = self.hyper_params['uniform_lambda_L2']*cur_lr, Y_full = Y_full, 
                                                target_min = 0, target_max = self.hyper_params['uniform_max']) # uniform distribution
        param_reg = self.mod.L2_reg(self.hyper_params['param_lambda_L2']) # all model weights and signaling network biases

#             total_loss = fit_loss + sign_reg + ligand_reg + param_reg + stability_loss + uniform_reg
        total_loss = fit_loss + sign_reg + param_reg + stability_loss + uniform_reg

        # gradient
        total_loss.backward()
        self.mod.add_gradient_noise(noise_level = self.hyper_params['gradient_noise_scale'])
        self.prediction_optimizer.step()

        # store
        cur_eig.append(spectral_radius)
        cur_loss.append(fit_loss.item())

    self.stats = utils.update_progress(self.stats, iter = e, loss = cur_loss, eig = cur_eig, learning_rate = cur_lr, 
                                n_sign_mismatches = self.mod.signaling_network.count_sign_mismatch())

    if verbose and e % (self.hyper_params['max_epochs']/100) == 0:
        utils.print_stats(self.stats, iter = e)

    if np.logical_and(e % self.hyper_params['reset_optimizer_epoch'] == 0, e>0):
        self.prediction_optimizer.state = self.reset_state.copy()

if verbose:
    mins, secs = divmod(time.time() - start_time, 60)
    print("Training ran in: {:.0f} min {:.2f} sec".format(mins, secs))

In [24]:
self.discriminator_loss

{'TF_clusters': BCEWithLogitsLoss(), 'celltype': CrossEntropyLoss()}

torch.nn.modules.loss.BCEWithLogitsLoss

# End dev

In [ ]:
# mod.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
# mod.signaling_network.prescale_weights(target_radius = target_spectral_radius) # spectral radius

# # trainer = TrainSimple(mod = mod,
# #                       optimizer = torch.optim.Adam,
# #                       prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
# #                       hyper_params = training_params,
# #                       train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
# #                       train_seed = seed)
# # res = trainer.train_model()

# trainer = TrainCat(mod = mod,
#                       optimizer = torch.optim.Adam,
#                       prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
#                       hyper_params = training_params,
#                       train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
#                       train_seed = seed)
# res = trainer.train_model()